In [6]:
import os, glob
if not os.path.exists("original datasets"):
    !git clone https://github.com/prajwalS-07/Default-of-Credit-Card-Clients.git
    repo = glob.glob("Default-of-Credit-Card-Client*")[0]
    %cd {repo}
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns

In [ ]:
##Loading the Raw Dataset

In [7]:
path = glob.glob("original datasets/*")[0]
print("Using file:", path)
df = pd.read_csv(path) if path.endswith(".csv") else pd.read_excel(path, header=1)
if "LIMIT_BAL" not in df.columns:
    df.columns = df.iloc[0]; df = df[1:].astype(float)
df = df.rename(columns={"PAY_1": "PAY_0",
                        "default payment next month": "default",
                        "default.payment.next.month": "default"})
print(df.shape)
df.head()

Using file: original datasets/default of credit card clients.xls
(30000, 25)


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [ ]:
##Missing vales

In [8]:
print("Total missing:", df.isnull().sum().sum())
df.isnull().sum()[df.isnull().sum() > 0]

Total missing: 0


,0


In [ ]:
##Duplicate record and identifier issues

In [9]:
print("Duplicate IDs      :", df["ID"].duplicated().sum() if "ID" in df.columns else "no ID column")
feat = df.drop(columns=["ID"], errors="ignore")
print("Duplicate full rows :", feat.duplicated().sum())
before = len(df)
df = df.drop(columns=["ID"], errors="ignore").drop_duplicates().reset_index(drop=True)
print("Rows removed:", before - len(df), "| rows now:", len(df))

Duplicate IDs      : 0
Duplicate full rows : 35
Rows removed: 35 | rows now: 29965


In [ ]:
##validate categorical codes against documentation

In [10]:
for c in ["SEX", "EDUCATION", "MARRIAGE"]:
    print(c, df[c].value_counts().sort_index().to_dict())
print("PAY_0", df["PAY_0"].value_counts().sort_index().to_dict())

SEX {1: 11874, 2: 18091}
EDUCATION {0: 14, 1: 10563, 2: 14019, 3: 4915, 4: 123, 5: 280, 6: 51}
MARRIAGE {0: 54, 1: 13643, 2: 15945, 3: 323}
PAY_0 {-2: 2750, -1: 5682, 0: 14737, 1: 3667, 2: 2666, 3: 322, 4: 76, 5: 26, 6: 11, 7: 9, 8: 19}


In [11]:
# Undocumented codes are grouped into "others"
df["EDUCATION"] = df["EDUCATION"].replace({0: 4, 5: 4, 6: 4})
df["MARRIAGE"] = df["MARRIAGE"].replace({0: 3})
for c in ["EDUCATION", "MARRIAGE"]:
    print(c, df[c].value_counts().sort_index().to_dict())

EDUCATION {1: 10563, 2: 14019, 3: 4915, 4: 468}
MARRIAGE {1: 13643, 2: 15945, 3: 377}


In [ ]:
##Impossible or suspicious age,credit-limit and amount values

In [12]:
bill = [f"BILL_AMT{i}" for i in range(1, 7)]
pay  = [f"PAY_AMT{i}" for i in range(1, 7)]
print("Age range           :", df["AGE"].min(), "to", df["AGE"].max())
print("LIMIT_BAL <= 0      :", (df["LIMIT_BAL"] <= 0).sum())
print("LIMIT_BAL range     :", df["LIMIT_BAL"].min(), "to", df["LIMIT_BAL"].max())
print("Rows with negative bills   :", (df[bill] < 0).any(axis=1).sum())
print("Rows with negative payments:", (df[pay] < 0).any(axis=1).sum())
print("Bills above credit limit   :", (df[bill].max(axis=1) > df["LIMIT_BAL"]).sum())

Age range           : 21 to 79
LIMIT_BAL <= 0      : 0
LIMIT_BAL range     : 10000 to 1000000
Rows with negative bills   : 1930
Rows with negative payments: 0
Bills above credit limit   : 3931


In [ ]:
##Are Zero values valid or hidden missing values

In [13]:
print("Rows with all 6 payments = 0:", (df[pay] == 0).all(axis=1).sum())
print("Rows with all 6 bills = 0    :", (df[bill] == 0).all(axis=1).sum())
print("Zero payments:", (df[pay] == 0).sum().sum(), "of", df[pay].size, "values")

Rows with all 6 payments = 0: 1401
Rows with all 6 bills = 0    : 836
Zero payments: 36711 of 179790 values


In [ ]:
##Save the clean dataset

In [14]:
df.to_csv("cleaned_data.csv", index=False)
print("Saved cleaned_data.csv", df.shape)
print("Download it: folder icon on the left, three dots, Download")

Saved cleaned_data.csv (29965, 24)
Download it: folder icon on the left, three dots, Download


In [ ]:
from google.colab import drive
drive.mount('/content/drive')